In [1]:
import numpy as np

from neuralnetlib.models import Transformer
from neuralnetlib.preprocessing import Tokenizer, pad_sequences
from neuralnetlib.losses import CrossEntropyWithLabelSmoothing
from neuralnetlib.optimizers import Adam
from neuralnetlib.callbacks import EarlyStopping, Callback, LearningRateScheduler

In [2]:
def prepare_causal_lm_data(text_data, tokenizer, context_length=32, target_length=16, stride=16):
    # The Transformer of the library is an encoder-decoder: the encoder reads a context and the decoder learns to
    # generate its continuation. The continuation must not be part of the encoder input (otherwise the model only
    # learns to copy it), and the model adds the <SOS>/<EOS> tokens to the decoder sequences itself.
    tokens = tokenizer.texts_to_sequences([text_data], add_special_tokens=False)[0]

    X, y = [], []
    for i in range(0, len(tokens) - context_length - target_length + 1, stride):
        X.append(tokens[i:i + context_length])
        y.append(tokens[i + context_length:i + context_length + target_length])

    return np.array(X), np.array(y)

In [3]:
class TextGenerationCallback(Callback):
    def __init__(self, model, tokenizer, prompt_texts, max_length=20, temperature=0.8):
        self.model = model
        self.tokenizer = tokenizer
        self.prompt_texts = prompt_texts
        self.max_length = max_length
        self.temperature = temperature
    
    def on_epoch_end(self, epoch, logs=None):
        print(f"\nGénération d'exemples après l'epoch {epoch + 1}:")
        for prompt in self.prompt_texts:
            # the prompts are encoded like the training contexts (without special tokens)
            sequence = self.tokenizer.texts_to_sequences([prompt], add_special_tokens=False)[0]
            input_sequence = pad_sequences([sequence], max_length=self.model.max_sequence_length, 
                                        padding='post', pad_value=self.model.PAD_IDX)
            
            generated = self.model.predict(
                input_sequence,
                max_length=self.max_length,
                temperature=self.temperature,
                beam_size=1
            )
            
            generated_text = self.tokenizer.sequences_to_texts(generated.tolist())[0]
            print(f"Prompt: {prompt}")
            print(f"Généré: {generated_text}\n")

In [4]:
def train_causal_lm(text_data, model, tokenizer, context_length=32, target_length=16, batch_size=32, epochs=10):
    X, y = prepare_causal_lm_data(text_data, tokenizer, context_length, target_length)
    print(f"{len(X)} training sequences (context: {X.shape[1]} tokens, continuation: {y.shape[1]} tokens)")

    # text8 is an English corpus: the prompts must use its vocabulary
    test_prompts = [
        "the history of",
        "in the early",
        "one of the most",
        "the city of"
    ]
    
    callbacks = [
        EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
        TextGenerationCallback(model, tokenizer, test_prompts),
        LearningRateScheduler(
            schedule="warmup_cosine",
            initial_learning_rate=0.001,
            # the cosine decay must end with the training
            schedule_params={'warmup_epochs': 1, 'total_epochs': epochs},
            verbose=True
        )
    ]
    
    history = model.fit(
        X, y,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
    )
    
    return history

In [5]:
def generate_text(model, tokenizer, prompt, max_length=20, temperature=0.8):
    sequence = tokenizer.texts_to_sequences([prompt], add_special_tokens=False)[0]
    input_sequence = pad_sequences([sequence], max_length=model.max_sequence_length, 
                                 padding='post', pad_value=model.PAD_IDX)
    
    generated = model.predict(
        input_sequence,
        max_length=max_length,
        temperature=temperature,
        beam_size=1
    )
    
    return tokenizer.sequences_to_texts(generated.tolist())[0]

In [6]:
with open('text8_light.txt', 'r', encoding='utf-8') as f:
    text_data = f.read()

# A part of the corpus and its most frequent words (the other ones become <UNK>), so that the training stays
# reasonably fast with NumPy
text_data = ' '.join(text_data.split()[:60000])

tokenizer = Tokenizer(filters='', num_words=3000)
tokenizer.fit_on_texts([text_data])

vocab_size = tokenizer.get_vocab_size()

In [7]:
CONTEXT_LENGTH = 32
TARGET_LENGTH = 16

model = Transformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=128,
    n_heads=4,
    n_encoder_layers=2,
    n_decoder_layers=2,
    d_ff=512,
    dropout_rate=0.1,
    max_sequence_length=max(CONTEXT_LENGTH, TARGET_LENGTH + 1),  # +1 for the <SOS>/<EOS> token of the decoder
    random_state=42
)

model.compile(
    loss_function=CrossEntropyWithLabelSmoothing(label_smoothing=0.1),
    optimizer=Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.98,
        epsilon=1e-9,
        clip_norm=1.0
    )
)

In [8]:
history = train_causal_lm(text_data, model, tokenizer, context_length=CONTEXT_LENGTH, target_length=TARGET_LENGTH,
                          epochs=10)

3748 training sequences (context: 32 tokens, continuation: 16 tokens)
Initial learning rate: 0.001000
[==============================] 100% Epoch 1/10 - loss: 6.2126 - 47.34s        
Génération d'exemples après l'epoch 1:
Prompt: the history of
Généré: the the the the the the the the the

Prompt: in the early
Généré: the the the the the the the the the

Prompt: one of the most
Généré: the the the the the the the the the

Prompt: the city of
Généré: the the the the the the the the the


[==============================] 100% Epoch 2/10 - loss: 5.7184 - 50.19s        
Génération d'exemples après l'epoch 2:
Prompt: the history of
Généré: the first of the first of the first of

Prompt: in the early
Généré: the first of the first of the first is

Prompt: one of the most
Généré: the war of the first of the first is

Prompt: the city of
Généré: the first of the first of the first is



Epoch 3: Learning rate adjusted to 0.000970
[==============================] 100% Epoch 3/10 - loss: 5.4443 -

In [9]:
generated = generate_text(model, tokenizer, "this would ensure that", temperature=0.8)
print(generated)

the world s own thought is not not not
